# Build extractor

In [18]:
from pydantic import BaseModel, Field, EmailStr
from typing import Optional, Literal
from pydantic_ai import Agent
from constants import MODEL_MEDIUM, MODEL_LARGE

class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["billing", "damaged product", "technical", "other"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(description="Summarize the email in 3-4 sentences.")

email_extractor_agent = Agent(MODEL_LARGE, system_prompt="""
    You are customer support agent, your task is to extract relevant information from an email.
""", output_type=EmailExtractor, retries=1)

In [4]:
import pandas as pd

df = pd.read_json("cleaned_data.json")

sample_mail = df.iloc[2]["inputs"]["email"]
sample_mail

"From: Oscar Johansson <oscar.johansson@yahoo.se>\nSubject: Cannot access my account for 3 days - Urgent help needed\n\nHello Support Team,\n\nI am reaching out because I have been completely locked out of my account for the past three days and I am running out of ideas on how to fix this on my own. The problem started on Monday evening when I tried to log in as usual but kept receiving an 'Invalid credentials' error despite being absolutely certain that I was entering the correct password.\n\nI followed the instructions on your website to reset my password, but the problem is that the password reset email never arrives in my inbox. I have checked my spam and junk folders multiple times, and there is nothing there either. I have attempted the reset process at least six or seven times across different browsers and even from my phone, but the result is always the same - no email arrives.\n\nThis is causing me real problems because I have important documents and data stored in my account 

In [5]:
result = await email_extractor_agent.run(sample_mail)

In [6]:
result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary="Oscar Johansson has been locked out of his account for three days, receiving 'Invalid credentials' errors. Password reset emails are not arriving in his inbox or spam folder despite multiple attempts. He urgently needs access to important work documents before a Friday deadline.")

In [7]:
result.output.summary

"Oscar Johansson has been locked out of his account for three days, receiving 'Invalid credentials' errors. Password reset emails are not arriving in his inbox or spam folder despite multiple attempts. He urgently needs access to important work documents before a Friday deadline."

TODO: show ground truth and compare

## Load in prompts from MLFlow

In [8]:
from mlflow.genai import load_prompt

class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["billing", "damaged product", "technical", "other"]
    urgency: Literal["low", "medium", "high"] = Field(description=load_prompt("email-urgency-description").format())
    summary: str = Field(description=load_prompt("summary_description").format(num_sentences = 4)
)

email_extractor_agent = Agent(MODEL_LARGE, system_prompt=load_prompt("email-extractor-system-prompt").format(), output_type=EmailExtractor, retries=1)

# instead of .format() you can use .template -> difference is you can send in variable inside format()

/Users/omeraytug/Desktop/llmops/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
result = await email_extractor_agent.run(sample_mail)
result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary="Oscar Johansson has been locked out of his account for three days due to 'Invalid credentials' errors despite entering the correct password. He has attempted password reset multiple times but the reset emails do not arrive in his inbox, spam, or junk folders. This issue is preventing him from accessing important documents and data needed for a work deadline this Friday. He requests urgent assistance and is willing to verify his identity.")

## LLM Judge

1. Require data with columns: inputs, expectations, outputs
2. mlflow experiments

In [10]:
result.output.model_dump()

{'sender_name': 'Oscar Johansson',
 'sender_email': 'oscar.johansson@yahoo.se',
 'issue_category': 'technical',
 'urgency': 'high',
 'summary': "Oscar Johansson has been locked out of his account for three days due to 'Invalid credentials' errors despite entering the correct password. He has attempted password reset multiple times but the reset emails do not arrive in his inbox, spam, or junk folders. This issue is preventing him from accessing important documents and data needed for a work deadline this Friday. He requests urgent assistance and is willing to verify his identity."}

TODO: dataframe with inputs, expectations, outputs but only 1 row

In [14]:
df["outputs"] = [{},(),result.output.model_dump(),{}]
df

,inputs,expectations,outputs
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""sender_name"": ""Erik L...",{}
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""sender_name"": ""Maja B...",()
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ...","{'sender_name': 'Oscar Johansson', 'sender_ema..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""sender_name"": ""Linnea...",{}


In [15]:
df_sample = df.drop([0,1,3])
df_sample

,inputs,expectations,outputs
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ...","{'sender_name': 'Oscar Johansson', 'sender_ema..."


- Using LLM Judge to compare

In [17]:
from mlflow.genai.scorers import get_all_scorers

# get_all_scorers()

In [19]:
import mlflow.genai
from mlflow.genai.scorers import Correctness, Summarization, Completeness, Fluency
import mlflow

llm_judge = "openrouter:/nvidia/nemotron-3-super-120b-a12b:free"

with mlflow.start_run(run_name="email_extractor_evaluator"):
    mlflow.log_param("model_judge", llm_judge)
    mlflow.log_param("model", MODEL_LARGE)
    
    results = mlflow.genai.evaluate(
        data=df_sample,
        scorers=[
            Correctness(model=llm_judge),
            Summarization(model=llm_judge)
        ]
    )

results

2026/04/15 14:54:59 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
Evaluating:   0%|          | 0/1 [Elapsed: 01:21, Remaining: ?]


KeyboardInterrupt: 